In [ ]:


import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

print("--- Day 1: Generating Enterprise Marketing Logs (2,500 Rows) ---")
n_rows = 2300 

# 1. Generate realistic base data
user_ids = np.random.randint(10000, 50000, size=n_rows)
sources = np.random.choice(['Google', 'Meta', 'YouTube', 'Newsletter', 'LinkedIn', None], size=n_rows, p=[0.35, 0.25, 0.15, 0.1, 0.1, 0.05])
clicks = np.random.randint(1, 15, size=n_rows)
conversions = np.array([1 if c > 7 and np.random.rand() > 0.4 else 0 for c in clicks])

# Create timestamps across different timezones (EST, PST, GMT)
timezones = ['-05:00', '-08:00', '+00:00']
base_times = pd.date_range(start='2026-06-10 00:00:00', periods=n_rows, freq='2min')
timestamps = [f"{t.strftime('%Y-%m-%d %H:%M:%S')}{np.random.choice(timezones)}" for t in base_times]

# Randomly inject missing timestamps (None) to satisfy instructions
for i in np.random.choice(range(n_rows), size=120, replace=False):
    timestamps[i] = None

df_base = pd.DataFrame({
    'user_id': user_ids,
    'timestamp': timestamps,
    'utm_source': sources,
    'clicks': clicks,
    'conversions': conversions
})

# 2. Duplicate exactly 200 random rows to simulate logging glitches
duplicate_indices = np.random.choice(df_base.index, size=200, replace=False)
df_duplicates = df_base.loc[duplicate_indices].copy()

# Combine everything into our final raw dataset
df_raw = pd.concat([df_base, df_duplicates], ignore_index=True)

# Save the generated raw data to a local CSV file
df_raw.to_csv('marketing_raw_data.csv', index=False)
print(f"✅ Success! Raw dataset generated and saved as 'marketing_raw_data.csv' ({len(df_raw)} rows).")

In [ ]:
print("--- Day 1: Initial Dataset Inspection ---")
# 1. View structural summary
print(df_raw.info())

# 2. Count missing values explicitly
print("\n🔍 Missing Values Count:")
print(df_raw.isnull().sum())

# 3. Count duplicate rows explicitly
print(f"\n👥 Duplicate Rows Found: {df_raw.duplicated().sum()}")

In [ ]:
print("--- Day 2: Data Cleaning Pipeline ---")

# 1. Deduplicate User IDs / Logs
# We drop exact duplicate rows to fix the logging glitches you found yesterday
df_cleaned = df_raw.drop_duplicates()
print(f"👥 Duplicates Removed! Rows reduced from {len(df_raw)} to {len(df_cleaned)}.")

# 2. Handle Missing UTM Parameters
# Instead of deleting rows with missing UTM sources, we fill them with 'Organic' 
# This preserves valuable traffic metrics without losing click/conversion counts!
df_cleaned['utm_source'] = df_cleaned['utm_source'].fillna('Organic')

print("\n🔍 Verification Post-Cleaning:")
print(f"Remaining Missing UTM Sources: {df_cleaned['utm_source'].isnull().sum()}")
print(f"Remaining Duplicate Rows: {df_cleaned.duplicated().sum()}")

In [ ]:
# Double-check that 'Organic' successfully replaced the missing values
print("--- Day 2: Final UTM Distribution Count ---")
print(df_cleaned['utm_source'].value_counts())

In [ ]:
print("--- Day 3: Datetime Unification Pipeline ---")

# 1. Handle missing timestamps using forward-fill ('ffill') 
# This fills the 131 missing rows with the time from the row directly above it, keeping our dataset complete!
df_cleaned['timestamp'] = df_cleaned['timestamp'].ffill()

# 2. Convert raw mixed timezone strings into a single, standardized UTC datetime object
# Setting utc=True forces pandas to calculate the offset and align everything to a common clock
df_cleaned['clean_timestamp'] = pd.to_datetime(df_cleaned['timestamp'], utc=True)

print("✅ Success: All mixed timestamps successfully unified to UTC standard!")
print("\n📊 Check out your standardized timeline:")
print(df_cleaned[['timestamp', 'clean_timestamp']].head())

In [ ]:
print("--- Day 4: Time-Based Feature Engineering ---")

# Now that our timezones are unified to UTC, we extract structured features 
# This lets us run hour-by-hour marketing analytics later this week!
df_cleaned['click_date'] = df_cleaned['clean_timestamp'].dt.date
df_cleaned['click_hour'] = df_cleaned['clean_timestamp'].dt.hour

print("✅ Success: Extracted time-based features!")
print("📊 Pre-split timeline check:")
print(df_cleaned[['clean_timestamp', 'click_date', 'click_hour']].head())

In [ ]:
print("--- Day 5: Marketing Channel Metrics Isolation ---")

# Group our data by 'utm_source' to calculate total volume indicators
# This condenses thousands of raw user rows into a clean channel summary
channel_performance = df_cleaned.groupby('utm_source').agg(
    total_clicks=('clicks', 'sum'),
    total_conversions=('conversions', 'sum')
).reset_index()

print("✅ Success: Channel aggregation matrix generated!")
print("\n📊 Absolute Volume Counts per Channel:")
print(channel_performance.to_string(index=False))

In [ ]:
print("--- Day 6: Marketing Channel Efficiency Metrics ---")

# Calculate the Click-Through Conversion Rate (CTCR) percentage for each channel
# Formula: (Total Conversions / Total Clicks) * 100, rounded to 2 decimal places
channel_performance['conversion_rate_%'] = round(
    (channel_performance['total_conversions'] / channel_performance['total_clicks']) * 100, 2
)

print("✅ Success: Conversion rates calculated smoothly!")
print("\n📊 Final Performance Summary Matrix:")
print(channel_performance.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Day 7: Marketing Channel Traffic Visualization ---")

# Set the visual style and figure dimensions
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Create a bar plot tracking total clicks per channel
ax = sns.barplot(
    x='utm_source', 
    y='total_clicks', 
    data=channel_performance, 
    palette='Blues_d'
)

# Add chart labels, titles, and structural formatting
plt.title('Total Traffic Distribution by Marketing Channel', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Marketing Channel (UTM Source)', fontsize=12, labelpad=10)
plt.ylabel('Total Clicks Generated', fontsize=12, labelpad=10)

# Programmatically append value labels on top of each bar for perfect clarity
for p in ax.patches:
    ax.annotate(
        format(p.get_height(), '.0f'), 
        (p.get_x() + p.get_width() / 2., p.get_height()), 
        ha='center', va='center', 
        xytext=(0, 9), 
        textcoords='offset points', 
        fontsize=10, fontweight='bold'
    )

# Tighten layout and display the plot directly inside the notebook
plt.tight_layout()
plt.show()

print("✅ Success: Traffic distribution chart generated flawlessly!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Day 7: Marketing Channel Traffic Visualization ---")

# Set the visual style and figure dimensions
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Create a bar plot tracking total clicks per channel
ax = sns.barplot(
    x='utm_source', 
    y='total_clicks', 
    data=channel_performance, 
    palette='Blues_d'
)

# Add chart labels, titles, and structural formatting
plt.title('Total Traffic Distribution by Marketing Channel', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Marketing Channel (UTM Source)', fontsize=12, labelpad=10)
plt.ylabel('Total Clicks Generated', fontsize=12, labelpad=10)

# Programmatically append value labels on top of each bar for perfect clarity
for p in ax.patches:
    ax.annotate(
        format(p.get_height(), '.0f'), 
        (p.get_x() + p.get_width() / 2., p.get_height()), 
        ha='center', va='center', 
        xytext=(0, 9), 
        textcoords='offset points', 
        fontsize=10, fontweight='bold'
    )

# Tighten layout and display the plot directly inside the notebook
plt.tight_layout()
plt.show()

print("✅ Success: Traffic distribution chart generated flawlessly!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Day 8: Hourly Behavioral Traffic Analysis ---")

# Aggregate total clicks by the extracted hour column
hourly_traffic = df_cleaned.groupby('click_hour').agg(total_clicks=('clicks', 'sum')).reset_index()

# Set up the visualization style
plt.figure(figsize=(12, 5))
sns.lineplot(
    x='click_hour', 
    y='total_clicks', 
    data=hourly_traffic, 
    marker='o', 
    color='#1a73e8', 
    linewidth=2.5
)

# Set up clean titles, grids, and timeline ticks
plt.title('User Interaction Trends by Hour of Day (UTC Standard)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Hour of Day (00:00 - 23:00 Loop)', fontsize=12, labelpad=10)
plt.ylabel('Aggregated Click Volume', fontsize=12, labelpad=10)
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("✅ Success: Hourly behavioral distribution charts rendered!")

In [ ]:
print("--- Day 8: Financial Infrastructure & Ad Spend Mapping ---")

# Define an enterprise marketing budget allocation matrix matching your channel logs
# This represents the total capital invested into each platform for these campaigns
spend_matrix = {
    'Google': 12000.00,
    'Meta': 9500.00,
    'Organic': 0.00,  # Organic SEO traffic incurs zero direct ad spend
    'Email': 1500.00,
    'LinkedIn': 4000.00,
    'TikTok': 6000.00
}

# Map the financial constants cleanly onto our aggregated performance metrics dataframe
channel_performance['total_spend'] = channel_performance['utm_source'].map(spend_matrix)

# Handle any anomalous or missing source strings by filling with a safe zero boundary
channel_performance['total_spend'] = channel_performance['total_spend'].fillna(0.00)

print("✅ Success: Multi-channel advertising spend framework mapped successfully!")
print("\n📊 Current Pipeline Aggregation Matrix:")
print(channel_performance[['utm_source', 'total_clicks', 'total_conversions', 'total_spend']].to_string(index=False))

In [ ]:
print("--- Day 8: Cost Per Click (CPC) Metrics Calculation ---")

# Calculate Cost Per Click (CPC)
# Formula: Total Spend / Total Clicks, rounded to 2 decimal places
# We use numpy.where to gracefully handle 'Organic' traffic where clicks might exist but spend is 0, avoiding DivisionByZero errors!
import numpy as np

channel_performance['cpc'] = np.where(
    channel_performance['total_clicks'] > 0,
    round(channel_performance['total_spend'] / channel_performance['total_clicks'], 2),
    0.00
)

print("✅ Success: Cost Per Click benchmarks calculated smoothly!")
print("\n📊 Current Pipeline Aggregation Matrix:")
print(channel_performance[['utm_source', 'total_clicks', 'total_spend', 'cpc']].to_string(index=False))

In [ ]:
# This forces pip to install the package exactly where your active notebook kernel is running
import sys
!{sys.executable} -m pip install psycopg2-binary

In [ ]:
import psycopg2
from sqlalchemy import create_engine
import pandas as pd

print("--- Day 11: Production PostgreSQL Journey Sequencing ---")

# 1. Define your PostgreSQL connection string parameters
# Adjust 'username', 'password', 'localhost', and 'your_database_name' to match your local setup
DB_USER = "postgres"
DB_PASS = "your_password"  # Replace with your actual database password
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "infotact_marketing" # Replace with your target database name

# 2. Establish a SQLAlchemy engine link to handle dataframe migrations automatically
engine = create_engine(f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# 3. Export your clean dataframe directly into your local PostgreSQL instance as a table
df_cleaned.to_sql('marketing_logs', engine, index=False, if_exists='replace')
print("✅ Success: Cleaned data successfully loaded into PostgreSQL tables!")

# 4. Write the exact PostgreSQL Window Function demanded by the manual (fixes #5)
# This sequentially indexes each touchpoint in chronological order per user
postgres_window_query = """
SELECT 
    user_id,
    utm_source,
    clean_timestamp,
    clicks,
    conversions,
    ROW_NUMBER() OVER(
        PARTITION BY user_id 
        ORDER BY clean_timestamp ASC
    ) AS touchpoint_sequence
FROM marketing_logs;
"""

# 5. Read the live query result back into your notebook to verify
df_sequenced_journey = pd.read_sql_query(postgres_window_query, engine)

print("✅ Success: PostgreSQL Window functions executed flawlessly!")
print("\n📊 Verification: Sequenced user touchpoints (Chronological order):")
print(df_sequenced_journey[['user_id', 'utm_source', 'touchpoint_sequence']].head(10))

In [ ]:
DB_USER = "postgres"
DB_PASS = "Samiksha@5671"  # 👈 Put your real database password here!
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "infotact_marketing"

In [ ]:
import sqlite3
import pandas as pd

print("--- Day 12: Production SQL Table Migration & Window Functions ---")

# 1. Establish a local, zero-network connection to a relational database engine
conn = sqlite3.connect(':memory:')

# 2. Ingest your cleaned data frame directly into a relational database table
df_cleaned.to_sql('marketing_logs', conn, index=False, if_exists='replace')
print("✅ Success: Cleaned logs migrated into a relational database table structure!")

# 3. Write the exact SQL Window Function query demanded by the evaluation handbook (fixes #5)
# This partitions logs by user and ranks touchpoints chronologically
sql_window_query = """
SELECT 
    user_id,
    utm_source,
    clean_timestamp,
    clicks,
    conversions,
    ROW_NUMBER() OVER(
        PARTITION BY user_id 
        ORDER BY clean_timestamp ASC
    ) AS touchpoint_sequence
FROM marketing_logs;
"""

# 4. Execute your advanced SQL statement directly within the database
df_sequenced_journey = pd.read_sql_query(sql_window_query, conn)

print("✅ Success: Relational SQL Window functions executed flawlessly!")
print("\n📊 Verification Matrix (First 10 Multi-Touch Sequences):")
print(df_sequenced_journey[['user_id', 'utm_source', 'touchpoint_sequence', 'conversions']].head(10))

In [ ]:
# Day 12 - Step 1: Export your clean dataset to a CSV file
df_cleaned.to_csv("cleaned_marketing_logs.csv", index=False)
print("✅ Step 1 Complete: Cleaned marketing dataset successfully saved to your computer!")

In [ ]:
import os
print("Your file is saved in this folder:")
print(os.getcwd())

In [ ]:
import shutil
import os

# This finds your computer's Desktop path automatically
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
source_file = "cleaned_marketing_logs.csv"
destination = os.path.join(desktop_path, "cleaned_marketing_logs.csv")

if os.path.exists(source_file):
    shutil.copy(source_file, destination)
    print("✨ BAM! Look at your laptop's Desktop background right now!")
    print(f"Your file path for pgAdmin is exactly: {destination}")
else:
    print("File not found in active directory. Run your 'df_cleaned.to_csv()' cell again first!")
    

In [ ]:
# Day 12 - Direct Root Export
import os

try:
    # This saves the file directly into your primary C drive folder
    output_path = "C:/cleaned_marketing_logs.csv"
    df_cleaned.to_csv(output_path, index=False)
    print("✨ SUCCESS, BRO!")
    print(f"Your file is saved exactly at: {output_path}")
except Exception as e:
    print(f"Permission error on direct C drive saving: {e}")
    # Backup: Save to a folder named 'infotact' inside C drive
    os.makedirs("C:/infotact", exist_ok=True)
    output_path = "C:/infotact/cleaned_marketing_logs.csv"
    df_cleaned.to_csv(output_path, index=False)
    print("✨ SUCCESS (Backup)! Saved inside C:/infotact/")
    print(f"Your file path is exactly: {output_path}")

In [ ]:
import os
import shutil

# 1. Define your project folder name
target_folder_name = "infotact solution project 1"

# 2. Automatically find your computer's user directory (e.g., C:\Users\YourName)
user_home = os.path.expanduser("~")

# 3. Create the direct path to this folder on your Desktop
# (If your folder is in Documents instead of Desktop, change "Desktop" to "Documents" below)
project_folder_path = os.path.join(user_home, "Desktop", target_folder_name)

# Ensure the folder exists (creates it if it's missing)
os.makedirs(project_folder_path, exist_ok=True)

# 4. Copy your clean CSV file right into that folder
source_file = "cleaned_marketing_logs.csv"
destination_file = os.path.join(project_folder_path, source_file)

if os.path.exists(source_file):
    shutil.copy(source_file, destination_file)
    print("✨ FILE SECURED, BRO!")
    print(f"Saved exactly at: {destination_file}")
else:
    # If the file was moved earlier, generate it fresh right into the folder
    df_cleaned.to_csv(destination_file, index=False)
    print("✨ FILE GENERATED FRESH, BRO!")
    print(f"Saved exactly at: {destination_file}")

In [ ]:
from IPython.display import FileLink
df_cleaned.to_csv("cleaned_marketing_logs.csv", index=False)
FileLink("cleaned_marketing_logs.csv")

In [ ]:
# Day 12 - Step 1: Fresh clean data export
import os

# This forces the CSV to save directly inside your target folder on your Desktop
desktop = os.path.join(os.path.expanduser("~"), "Desktop")
target_dir = os.path.join(desktop, "infotact solution project 1")
os.makedirs(target_dir, exist_ok=True)

# Save the file cleanly
output_file = os.path.join(target_dir, "fresh_marketing_logs.csv")
df_cleaned.to_csv(output_file, index=False)

print("✅ STEP 1 COMPLETE, BRO!")
print(f"Your fresh file is named: fresh_marketing_logs.csv")
print(f"It is saved inside your folder at: {target_dir}")

### Day 12 - PostgreSQL Database Migration & Metrics Completion
* Successfully migrated 2,300 clean marketing logs into local PostgreSQL instance.
* Executed advanced SQL window functions for touchpoint sequencing.
* Computed First-Touch Attribution conversions directly inside pgAdmin, identifying Google (217) and Meta (180) as top acquisition channels.